In [1]:
import pandas as pd
input_df = pd.read_csv("/Users/balkrishna/Documents/Projects/market_place_360/campaign/export/sms_2326.csv", index_col=0)

input_df['campaign_type'] = 'Retail'

input_df.to_csv("/Users/balkrishna/Documents/Projects/market_place_360/campaign/export/sms_2326.csv", index=False)


In [7]:
#Define DB Variables
import psycopg2
DBNAME = "campaign_db"
USER = "balkrishna"
PASSWORD = "postgressql@14"
HOST = "localhost"

conn = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    password=PASSWORD,
    host=HOST
)

cur =conn.cursor()
# curr.execute("select ")

In [ ]:
from sqlalchemy import create_engine

DBNAME = "campaign_db"
USER = "balkrishna"
PASSWORD = "postgressql@14"
HOST = "localhost"

# engine = create_engine(
#     f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}/{DBNAME}"
# )

engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "dbname": DBNAME,
        "user": USER,
        "password": PASSWORD,
        "host": HOST
    }
)
db_df = pd.read_sql("""
    select customer_ref_no, first_name, last_name, email
    from cdm.seedlist_users
""", engine)


db_df.head()

,customer_ref_no,first_name,last_name,email
0,a6665b7b-c964-4a2c-b3b2-9bbe6cadf36a,Lisa,Randolph,graceabbott@example.com
1,d19f9e3f-4277-490f-8b2d-cff1265e7368,Alexis,Vega,bprince@example.org
2,a08c5d65-0113-4b3e-9336-b20d0ec4a886,Cheryl,Bush,jrodriguez@example.org
3,1a981db7-1ec0-4652-8817-d412094d0c2b,Jessica,Owens,theresacox@example.org
4,b7c80047-36f7-41a5-a4c1-6b3c1e821211,Christine,Hart,jillianpatterson@example.net


In [33]:
# we will build actuall seedlist customer database. 

# Additional Column list from the actual input file format which will be added in the seedlist file 

extra_cols = ['campaign_id','segment_id','execution_type',
              'campaign_execution_count','sms_template_id',
              'product_pitched']

for c in extra_cols:
    db_df[c] = input_df[c].iloc[0]
    
    

In [35]:
db_df.columns
db_df.head()

,customer_ref_no,first_name,last_name,email,campaign_id,segment_id,execution_type,campaign_execution_count,sms_template_id,product_pitched
0,a6665b7b-c964-4a2c-b3b2-9bbe6cadf36a,Lisa,Randolph,graceabbott@example.com,CAMP2326,SEG1234,SEEDLIST,1,promo_standard,CreditCard
1,1a981db7-1ec0-4652-8817-d412094d0c2b,Jessica,Owens,theresacox@example.org,CAMP2326,SEG1234,SEEDLIST,1,promo_standard,CreditCard
2,b7c80047-36f7-41a5-a4c1-6b3c1e821211,Christine,Hart,jillianpatterson@example.net,CAMP2326,SEG1234,SEEDLIST,1,promo_standard,CreditCard
3,82787f2a-1cd2-4c97-ab70-93520f7a1d23,Megan,Lopez,fowlerandrew@example.com,CAMP2326,SEG1234,SEEDLIST,1,promo_standard,CreditCard
4,c4b9555f-3f63-4d69-81f5-5a0d3d5abed2,Cody,Flowers,hchristensen@example.org,CAMP2326,SEG1234,SEEDLIST,1,promo_standard,CreditCard


In [35]:
import pandas as pd
input_df = pd.read_csv("/Users/balkrishna/Documents/Projects/market_place_360/campaign/export/email_2326.csv")

In [37]:
## another way fo using CONN connection

seedlist_customer=[]
campaign_type = input_df['campaign_type'].iloc[0].lower()
print(campaign_type)

db_df = pd.read_sql("""
            select customer_ref_no, first_name,last_name, email 
            from cdm.seedlist_users 
            where lower(segment)=%s 
            """
            ,engine
            ,params=(campaign_type,))

# db_df = pd.read_sql("""
#     select customer_ref_no, first_name,last_name,email
#     from cdm.seedlist_users
#     where lower(segment)=%s
# """, conn, params=(campaign_type,))

db_df.head()



retail


,customer_ref_no,first_name,last_name,email
0,a6665b7b-c964-4a2c-b3b2-9bbe6cadf36a,Lisa,Randolph,graceabbott@example.com
1,1a981db7-1ec0-4652-8817-d412094d0c2b,Jessica,Owens,theresacox@example.org
2,b7c80047-36f7-41a5-a4c1-6b3c1e821211,Christine,Hart,jillianpatterson@example.net
3,82787f2a-1cd2-4c97-ab70-93520f7a1d23,Megan,Lopez,fowlerandrew@example.com
4,c4b9555f-3f63-4d69-81f5-5a0d3d5abed2,Cody,Flowers,hchristensen@example.org


## Building Liverun Module

on the exclusion module, 
this is the table i created in excel which i am thinking to put as master file for rules. 
schema	table	column	exclusion_rule	exclusion_type	exclusion_name
cdm	customer_exclusion_master	reason_code	reason_code = 'DND'	MARKETING	Marketing OptOut
cdm	customer_exclusion_master	reason_code	reason_code = 'OPTOUT'	GLOBAL	Global OptOut
cdm	customer_exclusion_master	reason_code	reason_code = 'BLACKLIST'	GLOBAL	Blacklisted
cdm	customer_exclusion_master	reason_code	reason_code = 'NO_EMAIL'	CHANNEL	Email Optout

as of now i am not putting any exclusion column in it, because ideally there should be one
which will be used to decide which exclusion name and its rule needs to be picked from this sheet.
one more thing i have to keep on decreasing users after each exclusion in sequence not all in once. 
because we can't remove customer all in once that will be wrong logic. 


In [ ]:
# input_df['exclusion_rule'] = 'DND | Global_OptOut | Blacklisted | Email_Optout'

# input_df.to_csv("/Users/balkrishna/Documents/Projects/market_place_360/campaign/export/email_2326.csv", index=False)

# input_df = pd.read_csv("/Users/balkrishna/Documents/Projects/market_place_360/campaign/export/email_2326.csv")

input_df.head()

,customer_ref_no,first_name,last_name,email,gender,current_product,target_product,analyst_email,execution_type,template_id,campaign_id,segment_id,campaign_name,campaign_execution_count,product_pitched,campaign_type,channel,exclusion_rule
0,e90f7344-0ce5-4a43-abb9-fd8d2cfbf014,Marlin,Kinneally,mkinneally0@xrea.com,Male,Cinnamon Raisin Bread,Falafel Mix,balkrishna.11.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout
1,dcc1f144-443e-4d22-b94e-e142aed16ace,Donnie,MacCarrick,dmaccarrick1@amazon.de,Female,Kettle Corn Popcorn,Beef Stroganoff Mix,balkrishna.12.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout
2,4d41ad91-7348-4b26-926e-89e69dfb6c5f,Urson,Thody,uthody2@smh.com.au,Male,Cinnamon Sugar Donuts,Caramel Apple Chips,balkrishna.13.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout
3,478d6bc1-d514-45c7-8989-d44e600e43f3,Burke,Bodechon,bbodechon3@twitter.com,Male,Caramel Apple Taffy,Eco-Friendly Stainless Steel Straws,balkrishna.14.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout
4,824d2844-298e-4e81-9771-f4142b575f37,Yolande,Szach,yszach4@flavors.me,Female,Fridge Magnet Set,Digital Drawing Tablet,balkrishna.15.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout


In [110]:
import pandas as pd
exclusion_master_file = '/Users/balkrishna/Documents/Projects/market_place_360/campaign/config/exclusion_master.xlsx'
exclusion_mstr_df = pd.read_excel(exclusion_master_file)
print(exclusion_mstr_df.head())

# input file 
input_df = pd.read_csv("/Users/balkrishna/Documents/Projects/market_place_360/campaign/export/email_2326.csv")
# extracting exclusions from the input file 
exclusion_rules = input_df['exclusion_rule'].iloc[0].split("|")
print(exclusion_rules)
# creating a dedicated list values
# rules_list = exclusion_rules.split("|")

exclusion_rules = input_df['exclusion_rule'].iloc[0].split("|") # output : ['DND ', ' Global_OptOut ', ' Blacklisted ', ' Email_Optout']
rules_map = exclusion_mstr_df.set_index('exclusion_name')

for rule in exclusion_rules:
    table_name = rules_map.loc[rule.strip(), 'table']
    schema_name = rules_map.loc[rule.strip(), 'schema']
    exclusion_rule = rules_map.loc[rule.strip(), 'exclusion_rule']
    exclude_col = rules_map.loc[rule.strip(), 'column']
    
print(table_name)
print(schema_name)
print(exclusion_rule)
print(exclude_col)
    
    

  schema                      table       column             exclusion_rule  \
0    cdm  customer_exclusion_master  reason_code        reason_code = 'DND'   
1    cdm  customer_exclusion_master  reason_code     reason_code = 'OPTOUT'   
2    cdm  customer_exclusion_master  reason_code  reason_code = 'BLACKLIST'   
3    cdm  customer_exclusion_master  reason_code   reason_code = 'NO_EMAIL'   

  exclusion_type exclusion_name  
0      MARKETING            DND  
1         GLOBAL  Global_OptOut  
2         GLOBAL    Blacklisted  
3        CHANNEL   Email_Optout  
['DND ', ' Global_OptOut ', ' Blacklisted ', ' Email_Optout']
customer_exclusion_master
cdm
reason_code = 'NO_EMAIL'
reason_code


In [85]:
rules_map = exclusion_mstr_df.set_index('exclusion_name')

In [86]:
rule = 'DND'
table_name = rules_map.loc[rule, 'table']
schema_name = rules_map.loc[rule, 'schema']
exclusion_rule = rules_map.loc[rule, 'exclusion_rule']
exclude_col = rules_map.loc[rule, 'column']


print(table_name, schema_name, exclusion_rule, exclude_cond)

customer_exclusion_master cdm reason_code = 'DND' reason_code


In [ ]:


from datetime import datetime
excluded_rows = pd.DataFrame()
curr_schema_name = 'clm'
curr_table_nm = 'export_'+ datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
# insert Data back to DB as temp 
input_df.to_sql(
    name=curr_table_nm,
    con=engine,
    if_exists="replace",
    index=False,
    method="multi",
    schema="clm"
)



In [88]:
from datetime import datetime
excluded_rows = pd.DataFrame()
curr_schema_name = 'clm'
curr_table_nm = 'export_'+ datetime.now().strftime("%Y_%m_%d") # replace with time once finalzied strftime("%Y_%m_%d_%H_%M_%S")
# insert Data back to DB as temp 
input_df.to_sql(
    name=curr_table_nm,
    con=engine,
    if_exists="replace",
    index=False,
    method="multi",
    schema="clm"
)

#excute to perform exclusion : extract excludede customers.
query = f"""
select cur.customer_ref_no
from {curr_schema_name}.{curr_table_nm} cur
where exists (
    select 1
    from {schema_name}.{table_name} cem
    where cem.customer_ref_no = cur.customer_ref_no
    and  {exclusion_rule}
)
"""

excluded_rows = pd.read_sql(query, engine)
excluded_rows.count()
excluded_rows.head()

# [parameters: ("reason_code = 'DND'",)]

,customer_ref_no
0,da0e2ea7-dca5-4063-9414-e6f0d6247347
1,98fad45f-853c-4f25-868a-7f7ab75cfcda
2,dd20f578-cae5-4640-b117-d5479d20dc49
3,ea4ce973-46c5-4900-b980-8b1bbd6dd471
4,a3e9acd8-5cbe-445d-ad15-b6abd4ed35a0


In [91]:
#filter Valid ids
excluded_ids = set(excluded_rows['customer_ref_no'])
valid_df = input_df[
    ~input_df['customer_ref_no'].isin(excluded_rows['customer_ref_no'])
    ]

input_count = len(input_df)
excluded_count = len(excluded_rows)
valid_count = len(valid_df)

if input_count == (excluded_count + valid_count):
    print("counts_match")
    excluded_rows['exclusion_name'] = rule
    excluded_rows['campaign_id'] = valid_df['campaign_id'].iloc[0]
    excluded_rows['created_at'] = datetime.now()
else:
    raise ValueError("Count mismatch ❌ Something wrong in filtering logic")

counts_match


In [90]:
len(valid_df)

907

In [93]:
excluded_rows.head()

,customer_ref_no,exclusion_name,campaign_id,created_at
0,da0e2ea7-dca5-4063-9414-e6f0d6247347,DND,CAMP2326,2026-03-05 14:37:35.108646
1,98fad45f-853c-4f25-868a-7f7ab75cfcda,DND,CAMP2326,2026-03-05 14:37:35.108646
2,dd20f578-cae5-4640-b117-d5479d20dc49,DND,CAMP2326,2026-03-05 14:37:35.108646
3,ea4ce973-46c5-4900-b980-8b1bbd6dd471,DND,CAMP2326,2026-03-05 14:37:35.108646
4,a3e9acd8-5cbe-445d-ad15-b6abd4ed35a0,DND,CAMP2326,2026-03-05 14:37:35.108646


In [94]:
excl_rows = pd.DataFrame(columns=['customer_ref_no','exclusion_name','campaign_id','created_at'])
excl_rows = pd.concat([excl_rows, excluded_rows], ignore_index=True)


/var/folders/67/2r4kphts20z15p4sm__6_yc40000gn/T/ipykernel_20006/3759650512.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  excl_rows = pd.concat([excl_rows, excluded_rows], ignore_index=True)


In [112]:
len(excl_rows)
# excl_rows.tail()

186

In [96]:
excl_rows = pd.concat([excl_rows, excluded_rows], ignore_index=True)

### Contact POlicy

In [114]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'col_name': [5, -1, -3],
    'col2': [2, 4, -3]
})

df['flag'] = np.select(
    [df['col_name'] <= 0, df['col2'] <= 0],
    ['CC', 'CO'],
    default='ELIGIBLE'
)

print(df)

   col_name  col2      flag
0         5     2  ELIGIBLE
1        -1     4        CC
2        -3    -3        CC


#### Contact Matching

In [121]:
import pandas as pd
input_df = pd.read_csv("/Users/balkrishna/Documents/Projects/market_place_360/campaign/export/email_2326.csv")

In [116]:
input_df['contact_source'] = 'CreditCard'

In [117]:
input_df.head()

,customer_ref_no,first_name,last_name,email,gender,current_product,target_product,analyst_email,execution_type,template_id,campaign_id,segment_id,campaign_name,campaign_execution_count,product_pitched,campaign_type,channel,exclusion_rule,contact_source
0,e90f7344-0ce5-4a43-abb9-fd8d2cfbf014,Marlin,Kinneally,mkinneally0@xrea.com,Male,Cinnamon Raisin Bread,Falafel Mix,balkrishna.11.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard
1,dcc1f144-443e-4d22-b94e-e142aed16ace,Donnie,MacCarrick,dmaccarrick1@amazon.de,Female,Kettle Corn Popcorn,Beef Stroganoff Mix,balkrishna.12.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard
2,4d41ad91-7348-4b26-926e-89e69dfb6c5f,Urson,Thody,uthody2@smh.com.au,Male,Cinnamon Sugar Donuts,Caramel Apple Chips,balkrishna.13.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard
3,478d6bc1-d514-45c7-8989-d44e600e43f3,Burke,Bodechon,bbodechon3@twitter.com,Male,Caramel Apple Taffy,Eco-Friendly Stainless Steel Straws,balkrishna.14.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard
4,824d2844-298e-4e81-9771-f4142b575f37,Yolande,Szach,yszach4@flavors.me,Female,Fridge Magnet Set,Digital Drawing Tablet,balkrishna.15.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard


In [119]:
input_df.to_csv("/Users/balkrishna/Documents/Projects/market_place_360/campaign/export/email_2326.csv", index=False)

In [122]:
input_df.head()

,customer_ref_no,first_name,last_name,email,gender,current_product,target_product,analyst_email,execution_type,template_id,campaign_id,segment_id,campaign_name,campaign_execution_count,product_pitched,campaign_type,channel,exclusion_rule,contact_source
0,e90f7344-0ce5-4a43-abb9-fd8d2cfbf014,Marlin,Kinneally,mkinneally0@xrea.com,Male,Cinnamon Raisin Bread,Falafel Mix,balkrishna.11.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard
1,dcc1f144-443e-4d22-b94e-e142aed16ace,Donnie,MacCarrick,dmaccarrick1@amazon.de,Female,Kettle Corn Popcorn,Beef Stroganoff Mix,balkrishna.12.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard
2,4d41ad91-7348-4b26-926e-89e69dfb6c5f,Urson,Thody,uthody2@smh.com.au,Male,Cinnamon Sugar Donuts,Caramel Apple Chips,balkrishna.13.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard
3,478d6bc1-d514-45c7-8989-d44e600e43f3,Burke,Bodechon,bbodechon3@twitter.com,Male,Caramel Apple Taffy,Eco-Friendly Stainless Steel Straws,balkrishna.14.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard
4,824d2844-298e-4e81-9771-f4142b575f37,Yolande,Szach,yszach4@flavors.me,Female,Fridge Magnet Set,Digital Drawing Tablet,balkrishna.15.shah@gmail.com,LIVERUN,promo_standard,CAMP2326,SEG1234,Promotional SMS campaign,1,CreditCard,Retail,EMAIL,DND | Global_OptOut | Blacklisted | Email_Optout,CreditCard


In [ ]:
input_2_df = input_df.drop('email', axis=1)

In [125]:
curr_contact_match = input_df['contact_source'].iloc[0]

In [126]:
print(curr_contact_match)

CreditCard
